# What is Convolution?

**Course:** IBM AI Engineering Professional Certificate  
**Module:** Deep Neural Networks with PyTorch  
**Author:** Jack Pumpuni Frimpong-Manso  
**Estimated Time:** 25 minutes

---

## Learning Objectives
After completing this lab you will be able to:
- Describe what convolution is and why it is used in image analysis
- Determine the size of a convolutional output using the standard formula
- Apply the stride parameter and explain how it affects output size
- Apply zero padding to control output dimensions

---

## Table of Contents
1. [Convolution Defined](#1.-Convolution-Defined)
2. [Determining the Size of Output](#2.-Determining-the-Size-of-Output)
3. [Stride Parameter](#3.-Stride-Parameter)
4. [Zero Padding](#4.-Zero-Padding)
5. [Practice Questions](#5.-Practice-Questions)
6. [Summary](#6.-Summary)

## Install and Import Libraries

In [ ]:
%%time
%pip install pandas numpy matplotlib scipy --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, misc

print(f"PyTorch version: {torch.__version__}")
print("All libraries imported successfully.")

---
## 1. Convolution Defined

Convolution is a **linear operation** — similar to a linear equation, dot product, or matrix multiplication — that is specially suited to image analysis because it:
- **Preserves spatial relationships** between neighbouring pixels
- **Requires fewer parameters** than fully-connected layers (parameter sharing)
- **Detects local features** (edges, textures) regardless of their position in the image

### Mathematical progression

| Operation | Formula |
|---|---|
| Linear equation (scalar) | $y = wx + b$ |
| Linear equation (vector) | $\mathbf{y} = \mathbf{w}\mathbf{x} + b$ |
| Matrix multiplication | $\mathbf{y} = \mathbf{w}\mathbf{X} + \mathbf{b}$ |
| **Convolution (tensor)** | $\mathbf{Y} = \mathbf{w} * \mathbf{X} + \mathbf{b}$ |

In convolution, the parameter **w** is called a **kernel** (or filter). It slides across the input image and performs element-wise multiplication followed by summation to produce each output value.

In [ ]:
# Create a 2D convolution object
# in_channels=1 (grayscale image), out_channels=1 (one feature map), kernel_size=3 (3x3 kernel)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
print("Convolution layer:", conv)
print("\nDefault (random) state dict:")
print(conv.state_dict())

In [ ]:
# Set the kernel to the Sobel-X operator (detects vertical edges)
# Sobel-X kernel:
#  [ 1,  0, -1]
#  [ 2,  0, -2]
#  [ 1,  0, -1]
conv.state_dict()['weight'][0][0] = torch.tensor([[1.0, 0, -1.0],
                                                   [2.0, 0, -2.0],
                                                   [1.0, 0.0, -1.0]])
conv.state_dict()['bias'][0] = 0.0

print("Kernel (Sobel-X):")
print(conv.state_dict()['weight'][0][0])
print("\nBias:", conv.state_dict()['bias'][0].item())

In [ ]:
# Create a dummy 5x5 grayscale image (1 sample, 1 channel, 5 rows, 5 cols)
# Tensor shape: (N, C, H, W) = (1, 1, 5, 5)
image = torch.zeros(1, 1, 5, 5)

# Set the middle column (index 2) to 1 — creates a vertical edge
image[0, 0, :, 2] = 1

print("Input image (5x5):")
print(image[0, 0])

In [ ]:
# Visualise the image
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(image[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input Image (5×5)\nVertical edge at column 2')
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Row')
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, f'{image[0,0,i,j].item():.0f}',
                     ha='center', va='center', color='red', fontsize=12)

kernel_vals = conv.state_dict()['weight'][0][0].detach().numpy()
axes[1].imshow(kernel_vals, cmap='RdBu', vmin=-2, vmax=2)
axes[1].set_title('Sobel-X Kernel (3×3)\nDetects vertical edges')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel_vals[i,j]:.0f}',
                     ha='center', va='center', color='black', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('convolution_image_kernel.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

In [ ]:
# Apply convolution
z = conv(image)
print("Output feature map z:")
print(z[0, 0].detach())
print(f"\nOutput shape: {z.shape}  →  (N=1, C_out=1, H_out=3, W_out=3)")
print("\nExplanation: The Sobel-X kernel responds strongly to the vertical edge.")
print("Positive values on the left of the edge, negative on the right.")

---
## 2. Determining the Size of Output

For a square input of size **M** and a square kernel of size **K**, the output size is:

$$M_{new} = M - K + 1$$

**Intuition:** The kernel must fit entirely within the image. Starting at position 0, the last valid starting position is at M-K, giving M-K+1 total positions.

In [ ]:
# Demonstrate output size formula
K = 2  # kernel size
M = 4  # image size

conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K)
conv1.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv1.state_dict()['bias'][0] = 0.0

image1 = torch.ones(1, 1, M, M)
print(f"Input image size:  M = {M}")
print(f"Kernel size:       K = {K}")
print(f"Expected output:   M_new = {M} - {K} + 1 = {M - K + 1}")

z1 = conv1(image1)
print(f"\nActual output z1:")
print(z1.detach())
print(f"Actual output shape: {z1.shape[2:4]}  ✓")

In [ ]:
# Visualise output size for different kernel sizes
M_fixed = 8
kernel_sizes = [2, 3, 4, 5]
output_sizes = [M_fixed - k + 1 for k in kernel_sizes]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(k) for k in kernel_sizes], output_sizes, color='steelblue', edgecolor='black')
for bar, size in zip(bars, output_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Kernel Size (K)', fontsize=12)
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size vs Kernel Size (M={M_fixed}, no padding, stride=1)', fontsize=13)
ax.axhline(y=M_fixed, color='red', linestyle='--', alpha=0.5, label=f'Input size = {M_fixed}')
ax.legend()
plt.tight_layout()
plt.savefig('output_size_vs_kernel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Stride Parameter

**Stride** controls how many pixels the kernel shifts at each step.  
With stride **s**, the output size formula becomes:

$$M_{new} = \left\lfloor \frac{M - K}{s} \right\rfloor + 1$$

A larger stride reduces the output size and is sometimes used as a downsampling alternative to pooling.

In [ ]:
# Stride = 2 example
stride = 2
K = 2
M = 4

conv3 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=stride)
conv3.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv3.state_dict()['bias'][0] = 0.0

expected = (M - K) // stride + 1
print(f"Input size M={M}, Kernel K={K}, Stride s={stride}")
print(f"Expected output: floor(({M}-{K})/{stride}) + 1 = {expected}")

z3 = conv3(image1)
print(f"\nOutput z3:")
print(z3.detach())
print(f"Actual shape: {z3.shape[2:4]}  ✓")

In [ ]:
# Compare output sizes across different strides
M_fixed = 8
K_fixed = 3
strides = [1, 2, 3, 4]
out_sizes = [(M_fixed - K_fixed) // s + 1 for s in strides]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(s) for s in strides], out_sizes, color='darkorange', edgecolor='black')
for bar, size in zip(bars, out_sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold')
ax.set_xlabel('Stride (s)', fontsize=12)
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size vs Stride (M={M_fixed}, K={K_fixed}, no padding)', fontsize=13)
plt.tight_layout()
plt.savefig('output_size_vs_stride.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Zero Padding

**Problem:** Repeated convolutions shrink the feature map rapidly. Boundary pixels are also processed far fewer times than central pixels, losing border information.

**Solution:** Add rows and columns of zeros around the image (**zero padding**).

With padding **p**, the effective image size becomes $M' = M + 2p$, and the output size is:

$$M_{new} = \left\lfloor \frac{M + 2p - K}{s} \right\rfloor + 1$$

**'Same' padding** (output size = input size) requires $p = \frac{K-1}{2}$ when stride = 1.

In [ ]:
# Without padding — non-integer result
K = 2
s = 3
M = 4

conv4 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=s)
conv4.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv4.state_dict()['bias'][0] = 0.0

raw_size = (M - K) / s + 1
print(f"Without padding: ({M}-{K})/{s} + 1 = {raw_size:.4f}  ← non-integer!")
print(f"PyTorch floors this to: {int(raw_size)}")

z4 = conv4(image1)
print(f"\nOutput z4:")
print(z4.detach())
print(f"Shape: {z4.shape[2:4]}")

In [ ]:
# With padding = 1 — recovers more output coverage
padding = 1
conv5 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K, stride=s, padding=padding)
conv5.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                    [1.0, 1.0]])
conv5.state_dict()['bias'][0] = 0.0

M_prime = M + 2 * padding
new_size = (M_prime - K) // s + 1
print(f"With padding={padding}: M' = {M} + 2×{padding} = {M_prime}")
print(f"Output size = ({M_prime}-{K})//{s} + 1 = {new_size}")

z5 = conv5(image1)
print(f"\nOutput z5:")
print(z5.detach())
print(f"Shape: {z5.shape[2:4]}  ✓")

In [ ]:
# Summary visualisation: how padding and stride interact with output size
M_fixed = 6
K_fixed = 3
configs = [
    {'stride': 1, 'padding': 0, 'label': 's=1, p=0'},
    {'stride': 1, 'padding': 1, 'label': 's=1, p=1 (same)'},
    {'stride': 2, 'padding': 0, 'label': 's=2, p=0'},
    {'stride': 2, 'padding': 1, 'label': 's=2, p=1'},
]
sizes = [(M_fixed + 2*c['padding'] - K_fixed) // c['stride'] + 1 for c in configs]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue', 'green', 'darkorange', 'purple']
bars = ax.bar([c['label'] for c in configs], sizes, color=colors, edgecolor='black')
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(size), ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.axhline(y=M_fixed, color='red', linestyle='--', alpha=0.6, label=f'Input size = {M_fixed}')
ax.set_ylabel('Output Size', fontsize=12)
ax.set_title(f'Output Size: Effect of Stride & Padding (M={M_fixed}, K={K_fixed})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('stride_padding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Practice Questions

### Q1: Kernel of zeros applied to a random image
**Question:** A kernel of all zeros (kernel_size=3) is applied to a 4×4 random image. What are the output values?

In [ ]:
# Q1: Kernel of all zeros
torch.manual_seed(42)
Image_q1 = torch.randn((1, 1, 4, 4))
print("Random input image:")
print(Image_q1[0, 0])

conv_q1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)
conv_q1.state_dict()['weight'][0][0] = torch.tensor([[0.0, 0.0, 0.0],
                                                      [0.0, 0.0, 0.0],
                                                      [0.0, 0.0, 0.0]])
conv_q1.state_dict()['bias'][0] = 0.0

z_q1 = conv_q1(Image_q1)
print("\nOutput (zero kernel):")
print(z_q1.detach())
print("\n✓ Answer: All output values are ZERO.")
print("  Reason: Every element-wise product with a zero kernel is 0.")
print(f"  Output shape: {z_q1.shape[2:]} = M - K + 1 = 4 - 3 + 1 = 2")

### Q2: Output size with kernel_size=2, stride=2

In [ ]:
# Q2: Compute output size analytically and verify
M_q2 = 4
K_q2 = 2
s_q2 = 2

M_new_q2 = (M_q2 - K_q2) // s_q2 + 1
print(f"Image size M = {M_q2}, Kernel K = {K_q2}, Stride s = {s_q2}")
print(f"Output size = ({M_q2} - {K_q2}) // {s_q2} + 1 = {M_new_q2}")

conv_q2 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=K_q2, stride=s_q2)
conv_q2.state_dict()['weight'][0][0] = torch.tensor([[1.0, 1.0],
                                                      [1.0, 1.0]])
conv_q2.state_dict()['bias'][0] = 0.0
image_q2 = torch.ones(1, 1, M_q2, M_q2)
z_q2 = conv_q2(image_q2)
print(f"\nVerification — actual output shape: {z_q2.shape[2:]}  ✓")
print(f"Output values: {z_q2.detach()}")

---
## 6. Summary

| Concept | Formula | Key Takeaway |
|---|---|---|
| Basic output size | $M_{new} = M - K + 1$ | Output is smaller than input |
| With stride | $M_{new} = \lfloor\frac{M-K}{s}\rfloor + 1$ | Larger stride → smaller output |
| With padding | $M_{new} = \lfloor\frac{M+2p-K}{s}\rfloor + 1$ | Padding recovers lost size |
| Same padding (s=1) | $p = \frac{K-1}{2}$ | Output = Input size |

In [ ]:
# Final comprehensive summary demo
def conv_output_size(M, K, s=1, p=0):
    """Calculate convolution output size given M, K, stride s, and padding p."""
    return (M + 2*p - K) // s + 1

print("=" * 55)
print("Convolution Output Size Calculator")
print("=" * 55)
test_cases = [
    (5, 3, 1, 0),  # basic
    (4, 2, 1, 0),  # basic
    (4, 2, 2, 0),  # stride
    (4, 2, 3, 1),  # stride + padding
    (6, 3, 1, 1),  # same padding
]
for M, K, s, p in test_cases:
    out = conv_output_size(M, K, s, p)
    print(f"M={M}, K={K}, s={s}, p={p}  →  output size = {out}")